# Import Modules and Create Spark Session

In [2]:
import json
import gzip
import pandas as pd
import numpy as np
import seaborn as sns
from urllib.request import urlopen
from pyspark.sql.functions import mean, col, udf, split, explode, array_remove, avg
from pyspark.sql.functions import concat_ws, size, count, countDistinct, log10, rand, rank, first
from pyspark.sql.functions import split, collect_set, collect_list, flatten
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, IndexToString, Word2Vec, VectorAssembler
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.stat import Summarizer
from pyspark.mllib.evaluation import RegressionMetrics
from sklearn.metrics.pairwise import cosine_similarity
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from pyspark.sql.window import Window
import string
import nltk
nltk.download('all')
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')
nltk.download('punkt')
from datetime import datetime

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package basque_grammars to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package basque_grammars is already up-to-date!
[nltk_data]    | Down

[nltk_data]    |   Package omw is already up-to-date!
[nltk_data]    | Downloading package omw-1.4 to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package omw-1.4 is already up-to-date!
[nltk_data]    | Downloading package opinion_lexicon to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package opinion_lexicon is already up-to-date!
[nltk_data]    | Downloading package panlex_swadesh to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package panlex_swadesh is already up-to-date!
[nltk_data]    | Downloading package paradigms to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package paradigms is already up-to-date!
[nltk_data]    | Downloading package pe08 to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package pe08 is already up-to-date!
[nltk_data]    | Downloading package perluniprops to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package pe

[nltk_data]    |   Package wordnet31 is already up-to-date!
[nltk_data]    | Downloading package wordnet_ic to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package wordnet_ic is already up-to-date!
[nltk_data]    | Downloading package words to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package words is already up-to-date!
[nltk_data]    | Downloading package ycoe to
[nltk_data]    |     /usr/local/share/nltk_data...
[nltk_data]    |   Package ycoe is already up-to-date!
[nltk_data]    | 
[nltk_data]  Done downloading collection all
[nltk_data] Downloading package punkt to /usr/local/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
spark = SparkSession \
    .builder \
    .appName('models') \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.kryoserializer.buffer.max", "10g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "30g") \
    .config("fraction spark.storage.memoryFraction", "0.8") \
    .config("spark.executor.memory", "20g") \
    .getOrCreate()

# Loading Dataset 

In [4]:
!wget --no-check-certificate https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Magazine_Subscriptions.json.gz

--2023-04-27 11:12:53--  https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Magazine_Subscriptions.json.gz
Resolving jmcauley.ucsd.edu (jmcauley.ucsd.edu)... 137.110.160.73
Connecting to jmcauley.ucsd.edu (jmcauley.ucsd.edu)|137.110.160.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12875273 (12M) [application/x-gzip]
Saving to: ‘Magazine_Subscriptions.json.gz.4’

Magazine_Subscripti 100%[===================>]  12.28M  8.13MB/s    in 1.5s    

2023-04-27 11:12:56 (8.13 MB/s) - ‘Magazine_Subscriptions.json.gz.4’ saved [12875273/12875273]



## With Hadoop (does not work on Sentiment)

In [5]:
# !hadoop fs -put Magazine_Subscriptions.json.gz /

In [6]:
# %%time
# data = spark.read.json('hdfs://st446-cluster-2-m:8051/Magazine_Subscriptions.json.gz')
# data.show(5)

## With Pandas (Works on all models)

In [17]:
%%time
### load the meta data
data = []
with gzip.open('Magazine_Subscriptions.json.gz') as f:
    for l in f:
        data.append(json.loads(l.strip()))
        
# convert list into pandas dataframe
df = pd.DataFrame.from_dict(data) 

# only keep columns we want
df = df[['overall', 'reviewerID', 'asin', 'reviewText']]

# Get subset of dataset
# df = df.sort_values(by="reviewerID")
# df = df[:1000]

# Define Schema
from pyspark.sql.types import *
schema = StructType([
    StructField("overall", DoubleType(), True),
    StructField("reviewerID", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("reviewText", StringType(), True),
])

reviews = spark.createDataFrame(df, schema=schema) 

CPU times: user 2.57 s, sys: 73.7 ms, total: 2.64 s
Wall time: 2.66 s


# Changing column names and split into train test

In [18]:
%%time
reviews = reviews.select(col("reviewerID"), col("asin"), col("reviewText"), col("overall"))\
                .withColumnRenamed("asin", "productID")\
                .withColumnRenamed("overall", "rating")

reviews_train, reviews_test = reviews.randomSplit([0.8, 0.2], seed=42)
reviews_test = reviews_test.drop("reviewText")
print("reviews_train shape: ", reviews_train.count())
print("reviews_test shape: ", reviews_test.count())

reviews_train shape:  71739
reviews_test shape:  17950
CPU times: user 10.5 ms, sys: 885 µs, total: 11.4 ms
Wall time: 626 ms


# Models

## 1. Baseline Model

In [9]:
%%time

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for i in partitions:
    
    start_time = datetime.now()
    
    reviews_train = reviews_train.repartition(i).cache()
    reviews_test = reviews_test.repartition(i).cache()

    alpha = reviews_train.select(mean("rating")).collect()[0][0]

    reviewer_means = reviews_train.groupBy("reviewerID")\
                                  .agg(avg("rating").alias("reviewer_mean"))

    product_means = reviews_train.groupBy("productID")\
                                 .agg(avg("rating").alias("product_mean"))

    reviews_test_with_means = reviews_test.join(reviewer_means, "reviewerID", "left")\
                                            .join(product_means, "productID", "left")\
                                            .fillna(alpha, subset=["reviewer_mean", "product_mean"])

    def baseline_model(row):
        reviewer_mean = row["reviewer_mean"]
        product_mean = row["product_mean"]
        baseline_rating = alpha + (reviewer_mean - alpha) + (product_mean - alpha)
        return (row["productID"], row["reviewerID"], row["rating"], baseline_rating)

    reviews_test_with_preds = reviews_test_with_means.rdd.map(baseline_model)
    reviews_test_with_preds = reviews_test_with_preds.toDF(["productID", "reviewerID", "rating", "baseline_pred"])

    # Getting Metrics
    baseline_val_and_preds = reviews_test_with_preds.select(["rating", "baseline_pred"])
    baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
    metrics = RegressionMetrics(baseline_val_and_preds)
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([metrics.meanSquaredError, metrics.rootMeanSquaredError, metrics.meanAbsoluteError, total_time])
    
    print(f'Partition {i} Done. Start Time = {start_time}, End Time = {end_time}.')

Partition 2 Done. Start Time = 2023-04-27 11:14:20.057726, End Time = 2023-04-27 11:14:24.865643.
Partition 4 Done. Start Time = 2023-04-27 11:14:25.319380, End Time = 2023-04-27 11:14:27.266737.
Partition 8 Done. Start Time = 2023-04-27 11:14:27.501625, End Time = 2023-04-27 11:14:29.216301.
Partition 16 Done. Start Time = 2023-04-27 11:14:29.547583, End Time = 2023-04-27 11:14:31.188712.
Partition 64 Done. Start Time = 2023-04-27 11:14:31.638298, End Time = 2023-04-27 11:14:34.389676.
Partition 128 Done. Start Time = 2023-04-27 11:14:35.438805, End Time = 2023-04-27 11:14:40.176257.
Partition 512 Done. Start Time = 2023-04-27 11:14:42.000903, End Time = 2023-04-27 11:14:54.695293.
CPU times: user 359 ms, sys: 93 ms, total: 452 ms
Wall time: 39.1 s


In [10]:
baseline_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
baseline_partitions

,MSE,RMSE,MAE,Time (s)
2,1.808252,1.344713,1.014084,4.807917
4,1.808252,1.344713,1.014084,1.947357
8,1.808252,1.344713,1.014084,1.714676
16,1.808252,1.344713,1.014084,1.641129
64,1.808252,1.344713,1.014084,2.751378
128,1.808252,1.344713,1.014084,4.737452
512,1.808252,1.344713,1.014084,12.694390


## 2. ALS

In [11]:
%%time
# get train and test
stringIndexer = StringIndexer(inputCol='reviewerID', outputCol='reviewer')
model_train_reviewer = stringIndexer.fit(reviews_train)
train = model_train_reviewer.transform(reviews_train)
model_test_reviewer = stringIndexer.fit(reviews_test)
test = model_test_reviewer.transform(reviews_test)

stringIndexer = StringIndexer(inputCol='productID', outputCol='product')
model_train_product = stringIndexer.fit(train)
train = model_train_product.transform(train)
model_test_product = stringIndexer.fit(test)
test = model_test_product.transform(test)

train = train.drop("reviewText").select('reviewer', 'product', 'rating')
test = test.drop("reviewText").select('reviewer', 'product', 'rating')

train.cache()
test.cache()

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for i in partitions:
    
    start_time = datetime.now()
    
    train = train.repartition(i).cache()
    test = test.repartition(i).cache()

    als = ALS(
        rank=10,
        maxIter=15,
        regParam=0.2,
        userCol='reviewer',
        itemCol='product',
        ratingCol='rating',  
        coldStartStrategy='drop',
        nonnegative=True
    )

    # fit the model
    model = als.fit(train)

    # performing predictions using the testing data
    prediction = model.transform(test)
    # prediction.cache().show()

    # evaluate the model by computing the RMSE on the test data
    mse_evaluator_als = RegressionEvaluator(metricName="mse", labelCol="rating", predictionCol="prediction")
    rmse_evaluator_als = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
    mae_evaluator_als = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

    mse = mse_evaluator_als.evaluate(prediction)
    rmse = rmse_evaluator_als.evaluate(prediction)
    mae = mae_evaluator_als.evaluate(prediction)
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([mse, rmse, mae, total_time])
    
    print(f'Partition {i} Done. Start Time = {start_time}, End Time = {end_time}.')

Partition 2 Done. Start Time = 2023-04-27 11:15:12.664575, End Time = 2023-04-27 11:15:37.783403.
Partition 4 Done. Start Time = 2023-04-27 11:15:37.783582, End Time = 2023-04-27 11:15:51.610811.
Partition 8 Done. Start Time = 2023-04-27 11:15:51.610962, End Time = 2023-04-27 11:16:05.411436.
Partition 16 Done. Start Time = 2023-04-27 11:16:05.411624, End Time = 2023-04-27 11:16:19.405177.
Partition 64 Done. Start Time = 2023-04-27 11:16:19.405818, End Time = 2023-04-27 11:16:34.882815.
Partition 128 Done. Start Time = 2023-04-27 11:16:34.883038, End Time = 2023-04-27 11:16:52.963574.
Partition 512 Done. Start Time = 2023-04-27 11:16:52.963739, End Time = 2023-04-27 11:17:17.062921.
CPU times: user 287 ms, sys: 57.1 ms, total: 344 ms
Wall time: 2min 17s


In [12]:
als_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
als_partitions

,MSE,RMSE,MAE,Time (s)
2,4.05929,2.014768,1.671666,25.118828
4,4.05929,2.014768,1.671666,13.827229
8,4.05929,2.014768,1.671666,13.800474
16,4.05929,2.014768,1.671666,13.993553
64,4.05929,2.014768,1.671666,15.476997
128,4.05929,2.014768,1.671666,18.080536
512,4.05929,2.014768,1.671666,24.099182


## 3. Similarity Based Model

In [26]:
%%time

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for j in partitions:
    
    start_time = datetime.now()
    
    reviews_train = reviews_train.repartition(i).cache()
    reviews_test = reviews_test.repartition(i).cache()
    
    # Data Preprocessing
    reviews_train_model = reviews_train.withColumn("words", split("reviewText", "\\.|!|\\?")) \
                                        .withColumn("words", explode("words")) \
                                        .withColumn("words", split("words", "\\s+")) \
                                        .withColumn("words", array_remove("words", ""))\
                                        .drop("reviewText")

    # Training Model (Set # of Partitions)
    word2Vec = Word2Vec(vectorSize=100, seed=42, numPartitions=16, inputCol="words", outputCol="model")
    sb_model = word2Vec.fit(reviews_train_model)

    # Fitting Data on Model
    vectors = sb_model.transform(reviews_train_model)
    vectors = vectors.drop("words")

    # Getting average word2vec vector scores for each rating (1 to 5) for relevant reviews & products (10 scores in total)
    # Add each score for each (reviewerID, productID) pair as an additional column - 10 new columns total
    assembler = VectorAssembler(inputCols=["model"], outputCol="features")
    vectors_assembled = assembler.transform(vectors)

    vectors_reviewers = vectors_assembled.groupBy('reviewerID', 'rating')\
                                        .agg(Summarizer.mean(col("features")).alias("avg_vector"))\
                                        .withColumnRenamed("rating", "rating_")

    vectors_products = vectors_assembled.groupBy('productID', 'rating')\
                                        .agg(Summarizer.mean(col("features")).alias("avg_vector"))\
                                        .withColumnRenamed("rating", "rating_")

    vectors_all = reviews_test
    for i in range(1,6):
        vectors_all = vectors_all.join(vectors_reviewers.filter(f"rating = {i}"), on='reviewerID', how='left')\
                                .withColumnRenamed('avg_vector', f"r_{i}")\
                                .drop("rating_")

        vectors_all = vectors_all.join(vectors_products.filter(f"rating = {i}"), on='productID', how='left')\
                                .withColumnRenamed('avg_vector', f"p_{i}")\
                                .drop("rating_")

    # Calculate cosine similarity scores and return predicted rating
    def cosine_similarity_map(row):
        cosine_scores = []

        for i in range(1,6):
            reviewer_vector = row[f"r_{i}"] #switch to index? row[3], row[5], ...
            product_vector = row[f"p_{i}"] #switch to index? row[3], row[5], ...

            if (reviewer_vector is None) or (product_vector is None):
                cosine_scores.append(None)

            else:
                cosine_score = cosine_similarity(reviewer_vector.reshape(1,100), product_vector.reshape(1,100))
                cosine_scores.append(cosine_score)


        try:
            predicted_rating = float(np.nanargmax(np.array(cosine_scores, dtype=np.float64))+1)
        except ValueError:
            predicted_rating = round(alpha,0)

        return (row[0], row[1], row["rating"], predicted_rating)


    vectors_all_pred = vectors_all.rdd.map(cosine_similarity_map)
    reviews_test_with_preds = vectors_all_pred.toDF(["productID", "reviewerID", "rating", "sb_pred"])

    # Getting Metrics
    baseline_val_and_preds = reviews_test_with_preds.select(["rating", "sb_pred"])
    baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
    metrics = RegressionMetrics(baseline_val_and_preds)
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([metrics.meanSquaredError, metrics.rootMeanSquaredError, metrics.meanAbsoluteError, total_time])
    
    print(f'Partition {j} Done. Start Time = {start_time}, End Time = {end_time}.')

Partition 2 Done. Start Time = 2023-04-27 11:59:57.762635, End Time = 2023-04-27 12:01:32.460314.
Partition 4 Done. Start Time = 2023-04-27 12:01:45.835610, End Time = 2023-04-27 12:02:37.781558.
Partition 8 Done. Start Time = 2023-04-27 12:02:50.864461, End Time = 2023-04-27 12:03:42.397554.
Partition 16 Done. Start Time = 2023-04-27 12:03:55.262874, End Time = 2023-04-27 12:05:16.719954.
Partition 64 Done. Start Time = 2023-04-27 12:05:43.226315, End Time = 2023-04-27 12:07:12.160716.
Partition 128 Done. Start Time = 2023-04-27 12:07:40.674963, End Time = 2023-04-27 12:09:14.061272.
Partition 512 Done. Start Time = 2023-04-27 12:09:42.310259, End Time = 2023-04-27 12:10:39.908775.
CPU times: user 1.06 s, sys: 271 ms, total: 1.33 s
Wall time: 10min 56s


In [28]:
similarity_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
similarity_partitions

,MSE,RMSE,MAE,Time (s)
2,1.913983,1.383468,1.021616,94.697679
4,1.910864,1.382340,1.021504,51.945948
8,1.913816,1.383407,1.021783,51.533093
16,1.909972,1.382017,1.020836,81.457080
64,1.911922,1.382723,1.021226,88.934401
128,1.910529,1.382219,1.020947,93.386309
512,1.911253,1.382481,1.021560,57.598516


## 4. Sentiment Based Model

In [ ]:
%%time

partitions = [2, 4, 8, 16, 64, 128, 512]
results = []

for i in partitions:
    
#     reviews_train, reviews_test = reviews.randomSplit([0.8, 0.2], seed=42)
#     reviews_test = reviews_test.drop("reviewText")
    
    start_time = datetime.now()
    
    reviews_train = reviews_train.repartition(i).cache()
    reviews_test = reviews_test.repartition(i).cache()
    
    #--------------------------------------------
    # Defining Data Cleaning Functions
    #--------------------------------------------

    # text pre-processing steps
    stop_words = set(stopwords.words('english'))
    table = str.maketrans('', '', string.punctuation)
    lmtzr = WordNetLemmatizer() 

    def preprocess_text(line):
        #parse data
        # get tokens from line
        tokens = word_tokenize(line)
        # convert to lower case
        tokens = [w.lower() for w in tokens]
        # remove punctuations from each word
        stripped = [w.translate(table) for w in tokens]
        # remove remaining tokens that are not alphabetic
        words = [word for word in stripped if word.isalpha()]
        # filter out stop words
        words = [w for w in words if not w in stop_words]
        # lemmatizing the words
        words = [lmtzr.lemmatize(w) for w in words]
        return ', '.join(words)


    # Convert preprocessing function to UDF
    preprocess_textUDF = udf(lambda x: preprocess_text(x), StringType())

    #--------------------------------------------
    # Extracting top 3 features from each reviewer
    #--------------------------------------------

    # Cleaning dataset for reviewer reviews
    reviewer_train = reviews_train.select("reviewerID", "reviewText")\
                                .withColumn("reviewWords", preprocess_textUDF(col("reviewText"))) 
    grouped = reviewer_train.drop("reviewText")

    grouped = grouped.select(
        *[col(c) for c in grouped.columns],
        split(col("reviewWords"),",").alias("words")).drop("reviewWords")

    # take original corpus of reviews from reviews_train
    grouped = grouped.groupBy("reviewerID")\
                    .agg(flatten(collect_list("words")).alias("words"))

    #tf
    reviews_corpus_total_terms = grouped.withColumn("total_terms_per_document", size("words"))

    reviews_corpus_term_count = reviews_corpus_total_terms.withColumn("word", explode("words"))\
                            .groupBy("reviewerID", "word")\
                            .agg(count("word").alias("term_count"))\
                            .join(reviews_corpus_total_terms.drop("words"), on="reviewerID", how='left')

    reviews_corpus_term_frequency = reviews_corpus_term_count.withColumn("term_frequency", col("term_count") / col("total_terms_per_document"))\
                                                            .drop("term_count", "total_terms_per_document")

    #idf
    reviews_corpus_total_documents = grouped.count()

    reviews_corpus_document_frequency = grouped.withColumn("word", explode('words'))\
                                            .groupBy('word')\
                                            .agg(countDistinct("reviewerID").alias("document_frequency"))

    reviews_corpus_idf = reviews_corpus_document_frequency.withColumn('idf', reviews_corpus_total_documents / (log10(col("document_frequency"))+1))\
                                                            .drop("document_frequency")

    #tfidf
    reviews_corpus_tfidf = reviews_corpus_term_frequency.join(reviews_corpus_idf, on="word", how='left')\
                                                        .withColumn("tfidf", col("term_frequency")*col("idf") + rand(seed=42) / 100)\
                                                        .drop("term_frequency", "idf")

    #top3
    window = Window.partitionBy("reviewerID").orderBy(col("tfidf").desc())
    features_top3 = reviews_corpus_tfidf.select("*", rank().over(window).alias("rank"))\
                                                        .filter(col("rank") <= 3)\
                                                        .groupBy("reviewerID")\
                                                        .pivot("rank").agg(first(col("word")))\
                                                        .select(col("reviewerID"),
                                                                col("1").alias("feature1"),
                                                                col("2").alias("feature2"),
                                                                col("3").alias("feature3"))

    #--------------------------------------------
    # Clean Reviews by Product
    #--------------------------------------------
    product_train = reviews_train.select("productID", "reviewText")
    product_train = product_train.withColumn("product_words", preprocess_textUDF(col("reviewText"))) 

    grouped_prod = product_train.drop("reviewText")
    grouped_prod = grouped_prod.select(
        *[col(c) for c in grouped_prod.columns],
        split(col("product_words"),",").alias("prod_words")).drop("product_words")

    product_train_new = grouped_prod.withColumn("prod_words", explode("prod_words"))\
                                    .groupBy("productID")\
                                    .agg(collect_list("prod_words"))\
                                    .withColumnRenamed('collect_list(prod_words)','product_words')

    #--------------------------------------------
    # Join Reviewer Features and Cleaned Reviews by product to reviews_test
    #--------------------------------------------
    reviews_test = reviews_test.join(features_top3, on="reviewerID", how='left')\
                                .join(product_train_new, on="productID", how='left')\
                                .drop("reviewer_words")

    #--------------------------------------------
    # Get sentiment scores from features in cleaned reviews
    #--------------------------------------------
    # UDFs
    def get_sentiment(text):
        sid = SentimentIntensityAnalyzer()
        return sid.polarity_scores(text)['compound']

    udf_get_sentiment = udf(get_sentiment, FloatType())

    def get_phrase_of_interest(text, feature):
        if text == None:
            return ''

        words_indices  = [i for i, x in enumerate(text) if x == feature]
        phrase_list = [''.join(text[max(0, i-5):min(len(text), i+6)]) for i in words_indices]
        phrase_combined = ' '.join(phrase_list)
        return phrase_combined

    udf_get_phrase_of_interest = udf(get_phrase_of_interest, StringType())

    reviews_test_2 = reviews_test.withColumn('phrase1', 
                                                 udf_get_phrase_of_interest(col("product_words"), col("feature1")))\
                                    .withColumn('phrase2', 
                                                udf_get_phrase_of_interest(col("product_words"), col("feature2")))\
                                    .withColumn('phrase3', 
                                                udf_get_phrase_of_interest(col("product_words"), col("feature3")))\
                                    .drop("product_words")

    reviews_test_3 = reviews_test_2.drop('feature1', 'feature2','feature3')\
                                    .withColumn('sentiment1', udf_get_sentiment(col('phrase1')))\
                                    .withColumn('sentiment2', udf_get_sentiment(col('phrase2')))\
                                    .withColumn('sentiment3', udf_get_sentiment(col('phrase3')))\

    reviews_test_4 = reviews_test_3.drop("phrase1","phrase2","phrase3")\
                                    .withColumn('sen_rating', (col('sentiment1') + col('sentiment2') + col('sentiment3')) * 2/3 + 3)

    #--------------------------------------------
    # Final Prediction
    #--------------------------------------------

    baseline_val_and_preds = reviews_test_4.select(["rating", "sen_rating"])
    baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
    metrics = RegressionMetrics(baseline_val_and_preds)

    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds()
    
    results.append([metrics.meanSquaredError, metrics.rootMeanSquaredError, metrics.meanAbsoluteError, total_time])
    
    print(f'Partition {i} Done. Start Time = {start_time}, End Time = {end_time}.')

Partition 2 Done. Start Time = 2023-04-27 11:42:38.189367, End Time = 2023-04-27 11:43:39.291008.
Partition 4 Done. Start Time = 2023-04-27 11:45:24.155230, End Time = 2023-04-27 11:46:08.775640.
Partition 8 Done. Start Time = 2023-04-27 11:47:44.615214, End Time = 2023-04-27 11:48:23.306807.
Partition 16 Done. Start Time = 2023-04-27 11:49:38.182235, End Time = 2023-04-27 11:50:11.027920.
Partition 64 Done. Start Time = 2023-04-27 11:51:47.917243, End Time = 2023-04-27 11:52:22.693744.
Partition 128 Done. Start Time = 2023-04-27 11:53:38.466148, End Time = 2023-04-27 11:54:20.123085.
Partition 512 Done. Start Time = 2023-04-27 11:55:36.028034, End Time = 2023-04-27 11:56:31.813444.
CPU times: user 1.72 s, sys: 607 ms, total: 2.33 s
Wall time: 15min 9s


In [25]:
sentiment_partitions = pd.DataFrame(results, columns=['MSE', 'RMSE', 'MAE', 'Time (s)'], index=partitions)
sentiment_partitions

,MSE,RMSE,MAE,Time (s)
2,2.816802,1.678333,1.521785,61.101641
4,2.814052,1.677514,1.520591,44.620410
8,2.814444,1.677631,1.521107,38.691593
16,2.814522,1.677654,1.521419,32.845685
64,2.814801,1.677737,1.521194,34.776501
128,2.813843,1.677451,1.521232,41.656937
512,2.816846,1.678346,1.521845,55.785410
